# Single-subject EEG analysis: sub-001, four-level n-back

This executable demo presents a reproducible, quality-controlled analysis of `sub-001` from the **Multimodal Cognitive Workload n-back Task, 4 Difficulties** dataset.

The analysis deliberately separates:

1. data-integrity and unit validation;
2. formal-trial behaviour;
3. EEG preprocessing and quality control;
4. descriptive spectral and event-related analyses.

> **Important design limitation.** The formal task used one contiguous block per level in the fixed order 1-back → 2-back → 3-back → 4-back. Therefore workload level is perfectly confounded with elapsed time, fatigue, electrode drift, and block order. EEG results below are descriptive. A workload classifier with ordinary random cross-validation would be misleading and is intentionally not reported.

Dataset: [OpenNeuro ds007169](https://openneuro.org/datasets/ds007169/versions/1.0.5)  
Conversion source: [QT-nback_study/conversion_package](https://github.com/LMBooth/QT-nback_study/tree/main/conversion_package)

## Setup and data discovery

Download ds007169 as described in [`data/README.md`](../data/README.md). The original XDF and BIDS event/channel tables are required. The CLI equivalent is `python scripts/run_subject.py --subject sub-001 --data-root data/ds007169`. No machine-specific path is assumed. Set `EEG_PIPELINE_ROOT` only if the notebook has been moved outside the repository.


In [ ]:
from pathlib import Path
import os
import sys
from IPython.display import display, Image

candidates = [Path(os.environ["EEG_PIPELINE_ROOT"]).expanduser()] if os.environ.get("EEG_PIPELINE_ROOT") else []
candidates += [parent for base in (Path.cwd(),) for parent in (base, *base.parents)]
ROOT = next((p.resolve() for p in candidates if (p / "src" / "eeg_pipeline").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the repository, or set EEG_PIPELINE_ROOT")
sys.path.insert(0, str(ROOT / "src"))
from eeg_pipeline import run_subject

SUBJECT = "sub-001"
DATA_ROOT = ROOT / "data" / "ds007169"
OUTPUT_ROOT = ROOT / "results"
CONFIG = ROOT / "config" / "default.yaml"


## 2. Read the original XDF and diagnose a BrainVision/XDF scaling inconsistency

The XDF metadata label the EEG stream values as **microvolts**. The public conversion script passes those numbers directly to `mne.io.RawArray`, although MNE expects EEG in **volts**. The supplied BrainVision binary is numerically consistent with a missing µV-to-V conversion: its stored counts are approximately \(10^7\) times the XDF numbers, as expected when values treated as volts are exported with 0.1 µV resolution.

This independent audit follows the XDF metadata and physiological amplitude plausibility. The original XDF is treated as the source of truth and converted with:

\[
x_{V} = (x_{XDF} - \mathrm{median}(x_{XDF})) \times 10^{-6}.
\]

Subtracting the channel median removes the large amplifier DC offset before filtering; it does not alter oscillatory power or ERP differences.
The optional BrainVision diagnostic is written to `unit_validation.csv`; all actual EEG analysis uses XDF. A different XDF unit stops the pipeline.


## 3. Behaviour and acquisition-dropout quality control

Only events between `started_n_back` and `finished_n_back` are treated as formal trials. This time-based definition is safer than relying on mixed Boolean/string representations of `istutorial`.

Signal-detection sensitivity uses the log-linear correction:

\[
H = \frac{hits+0.5}{targets+1}, \quad F = \frac{false\ alarms+0.5}{nonTargets+1}, \quad d'=\Phi^{-1}(H)-\Phi^{-1}(F).
\]

Because target prevalence differs strongly across levels, raw accuracy is not compared with a 50% “chance” line. Balanced accuracy and (d') are more informative about whether the participant discriminated targets from non-targets.

## 4. EEG preprocessing and channel quality control

Pipeline:

- original XDF EEG stream;
- microvolts → volts, after per-channel median removal;
- 0.5–40 Hz zero-phase FIR filter (the acquisition already used a 50 Hz notch);
- automated channel-range screen using median/MAD across channels;
- flag channels above robust z = 4 (F7 for sub-001), average-reference excluding flagged channels, then spherical-spline interpolation;
- no automated ICA: there is no dedicated EOG channel, so blind component removal would require manual validation and could remove frontal neural activity.

The robust channel range is the 1st-to-99th percentile after filtering, before re-referencing.

## 5. Descriptive spectral analysis

Each formal block is divided into non-overlapping 4 s windows. A window is retained when:

- reported data loss is at most 5 samples (20 ms; 0.5% of a 4 s window), and
- the maximum channel peak-to-peak amplitude is at most 600 µV.

Welch spectra use 2 s Hamming segments with 50% overlap. Band powers are normalized by 1–40 Hz power within the same channel and window. Regions of interest were specified before looking at condition differences:

- frontal-midline theta: Fz and Cz, 4–7 Hz;
- posterior alpha: P3, Pz, P4, O1, and O2, 8–13 Hz.

## 6. Descriptive stimulus-locked ERP

The task code displays each letter for 1 s and emits the trial marker when that letter is removed. Therefore letter onset is inferred as `marker onset − 1.0 s`.

Epochs span −0.2 to 1.0 s and use the prestimulus interval as baseline. An epoch is retained if it contains at most 3 reported dropped samples and its maximum channel peak-to-peak amplitude is at most 400 µV. This analysis is descriptive because condition and block order are confounded, and because the dataset contains no photodiode validation of screen timing.

## Run all stages

The call below performs XDF/timing and unit QC, formal-trial behavior, dropout assessment, preprocessing/channel QC, spectral windows, and descriptive ERPs. Tables and figures are saved under `results/sub-001/`. This notebook keeps the rationale alongside a single executable pipeline call.


In [ ]:
result = run_subject(SUBJECT, DATA_ROOT, OUTPUT_ROOT, CONFIG)
display(result["summary"])


## Inspect the QC and descriptive summaries

Check the unit comparison and channel flags for each new participant. Do not assume that F7 is bad in all recordings. Counts and rejection summaries should be reviewed before interpreting band power or ERP plots.


In [ ]:
for name in ("unit_validation.csv", "behaviour_by_level.csv", "dropout_summary.csv",
             "channel_qc.csv", "spectral_window_qc.csv", "spectral_summary.csv",
             "erp_epoch_counts.csv"):
    print("\n" + name)
    display(result["tables"][name].round(4))


In [ ]:
for filename in ("01_behaviour.png", "05_roi_bandpower.png", "06_psd_by_level.png",
                 "07_stimulus_locked_erp.png"):
    print(filename)
    display(Image(filename=str(result["output_dir"] / filename)))


## 7. Interpretation guardrails

### What this single-subject analysis supports

- The supplied archive is readable and contains synchronized task, EEG, ECG/misc, and pupil streams.
- Formal behaviour and acquisition dropout burden can be quantified reproducibly.
- After correcting the XDF microvolt-to-volt scale, filtering, interpolating F7, and applying explicit segment QC, interpretable EEG spectra and descriptive ERPs can be produced.
- The workflow is reusable for the remaining participants after confirming whether the same unit issue and channel-QC pattern apply.

### What it does **not** support

- It does not establish a population-level workload effect from one participant.
- It does not establish that any spectral difference was caused by n-back difficulty: level is perfectly confounded with time/order.
- It does not justify a valid four-class workload decoder. With one contiguous block per class, random train/test splitting would leak block-specific drift; block-wise cross-validation cannot leave out a level while still training that class.
- It does not justify selecting channels, frequency bands, or rejection thresholds post hoc to maximize a desired monotonic trend.

The correct next scientific step is to run the same prespecified pipeline across all included participants and use participant-level summaries. Any classifier should be framed as dataset-specific and validated with group-aware splits, while explicitly acknowledging the task-order confound.
### Technical scope

The original sub-001 notebook was run with data, whereas the published repository intentionally excludes raw recordings. `run_subject` handles different flagged channels and aligns ERP QC to MNE-selected events when boundary epochs are omitted. Review each new subject’s unit, timing and QC tables; a working pipeline is not automatic evidence that the design confound has been resolved.
